# Classificação com MobileNetV2 — Transfer Learning
### Aula 6 — Atividade 1 (Intensivo IA Cariri / PNAAT)

Projeto: [yolo-edge-api](https://github.com/kaypes/yolo-edge-api) — João Kayque Pereira de Souza

Este notebook treina um classificador de imagens usando **Transfer Learning** com
**MobileNetV2** pré-treinada no ImageNet, seguindo o roteiro da Aula 6. O dataset usado é o
[Cat and Dog](https://www.kaggle.com/datasets/tongpython/cat-and-dog) do Kaggle (2 classes),
baixado via `kagglehub`, exatamente como o exemplo do PDF da aula.

**Antes de rodar:** em `Ambiente de execução → Alterar o tipo de ambiente de execução`,
selecione **GPU (T4)** — sem isso o treino roda em CPU e fica bem mais lento.


## 1. Instalação das bibliotecas

In [ ]:
!pip install -q torch torchvision opencv-python matplotlib scikit-learn kagglehub


## 2. Baixando o dataset (Kaggle)

Datasets do Kaggle nem sempre seguem o mesmo padrão de pastas -- por isso inspecionamos a
estrutura baixada antes de apontar o `ImageFolder` para ela, como o próprio PDF da aula recomenda.


In [ ]:
import kagglehub

# Baixa a versão mais recente do dataset
path = kagglehub.dataset_download("tongpython/cat-and-dog")
print("Path to dataset files:", path)


In [ ]:
import os

for raiz, pastas, arquivos in os.walk(path):
    nivel = raiz.replace(path, '').count(os.sep)
    indent = '  ' * nivel
    print(f"{indent}{os.path.basename(raiz)}/")
    if nivel < 3:
        for arq in arquivos[:3]:
            print(f"{indent}  {arq}")


O dataset `tongpython/cat-and-dog` vem organizado como:

```
.../training_set/training_set/cats/
.../training_set/training_set/dogs/
.../test_set/test_set/cats/
.../test_set/test_set/dogs/
```

A rubrica da atividade exige explicitamente pastas físicas **`train/`** e **`validation/`**
com pelo menos 2 classes -- por isso, em vez de usar `torch.utils.data.random_split` num único
`ImageFolder` (como no exemplo genérico do PDF), materializamos essa estrutura em disco copiando
os arquivos do Kaggle para um layout próprio.

## 3. Organizando o dataset em `train/` e `validation/`

In [ ]:
import shutil
from pathlib import Path

SRC_TRAIN = Path(path) / "training_set" / "training_set"
SRC_TEST = Path(path) / "test_set" / "test_set"
DATASET_DIR = Path("dataset")
TRAIN_DIR = DATASET_DIR / "train"
VAL_DIR = DATASET_DIR / "validation"

CLASSES = ["cats", "dogs"]

if DATASET_DIR.exists():
    shutil.rmtree(DATASET_DIR)

for classe in CLASSES:
    (TRAIN_DIR / classe).mkdir(parents=True, exist_ok=True)
    (VAL_DIR / classe).mkdir(parents=True, exist_ok=True)

    for arq in (SRC_TRAIN / classe).glob("*.jpg"):
        shutil.copy(arq, TRAIN_DIR / classe / arq.name)

    for arq in (SRC_TEST / classe).glob("*.jpg"):
        shutil.copy(arq, VAL_DIR / classe / arq.name)

print("Estrutura organizada em:", DATASET_DIR.resolve())


### Estrutura final do dataset (entregável: captura da estrutura de pastas)

Célula abaixo imprime a árvore de diretórios com a contagem de imagens por classe --
**print obrigatório da atividade**.

In [ ]:
def imprime_estrutura(base_dir):
    base_dir = Path(base_dir)
    print(f"{base_dir}/")
    for split_dir in sorted(base_dir.iterdir()):
        if not split_dir.is_dir():
            continue
        print(f"  {split_dir.name}/")
        for classe_dir in sorted(split_dir.iterdir()):
            if not classe_dir.is_dir():
                continue
            n = len(list(classe_dir.glob("*.jpg")))
            print(f"    {classe_dir.name}/  ({n} imagens)")

imprime_estrutura(DATASET_DIR)


## 4. Importando bibliotecas

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
import numpy as np
import matplotlib.pyplot as plt


## 5. Pré-processamento e carregamento do dataset

In [ ]:
normalize = transforms.Normalize(
    mean=[0.485, 0.456, 0.406],
    std=[0.229, 0.224, 0.225]
)

data_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    normalize
])

train_dataset = datasets.ImageFolder(TRAIN_DIR, transform=data_transforms)
val_dataset = datasets.ImageFolder(VAL_DIR, transform=data_transforms)

print("Classes:", train_dataset.classes)
print("Imagens de treino:", len(train_dataset))
print("Imagens de validação:", len(val_dataset))

train_ds = torch.utils.data.DataLoader(train_dataset, batch_size=32, shuffle=True)
val_ds = torch.utils.data.DataLoader(val_dataset, batch_size=32, shuffle=False)


## 6. Carregando o modelo MobileNetV2 pré-treinado

In [ ]:
base_model = models.mobilenet_v2(
    weights=models.MobileNet_V2_Weights.IMAGENET1K_V1
)


## 7. Congelando as camadas do modelo base (Transfer Learning)

In [ ]:
for param in base_model.parameters():
    param.requires_grad = False


## 8. Construindo a nova camada de classificação

In [ ]:
num_classes = len(train_dataset.classes)

base_model.classifier = nn.Sequential(
    nn.Linear(base_model.last_channel, 128),
    nn.ReLU(),
    nn.Linear(128, num_classes)
)

model = base_model


## 9. Compilação do modelo (função de perda + otimizador)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
print("Dispositivo:", device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())


## 10. Treinamento do modelo (10 épocas)

### Entregável: captura de tela com a acurácia final após 10 épocas

In [ ]:
epochs = 10
history = {'accuracy': [], 'val_accuracy': [], 'loss': [], 'val_loss': []}

for epoch in range(epochs):
    model.train()
    correct, total, running_loss = 0, 0, 0.0
    for images, labels in train_ds:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

    train_acc = correct / total
    train_loss = running_loss / total

    model.eval()
    correct, total, running_val_loss = 0, 0, 0.0
    with torch.no_grad():
        for images, labels in val_ds:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            running_val_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs, 1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)

    val_acc = correct / total
    val_loss = running_val_loss / total

    history['accuracy'].append(train_acc)
    history['val_accuracy'].append(val_acc)
    history['loss'].append(train_loss)
    history['val_loss'].append(val_loss)

    print(f"Época {epoch+1}/{epochs} - loss: {train_loss:.4f} - acurácia treino: "
          f"{train_acc:.4f} - loss val: {val_loss:.4f} - acurácia validação: {val_acc:.4f}")

print()
print(f"Acurácia final de treino:    {history['accuracy'][-1]:.4f}")
print(f"Acurácia final de validação: {history['val_accuracy'][-1]:.4f}")


## 11. Avaliação do modelo — `model.evaluate(val_ds)`

A rubrica da atividade pede o resultado de `model.evaluate(val_ds)`, uma chamada da API do
Keras/TensorFlow. Como este notebook (assim como o restante do projeto) usa PyTorch, a função
abaixo replica o mesmo contrato do Keras -- recebe o modelo e o `val_ds`, roda uma passada de
avaliação e retorna `[loss, accuracy]` -- para que a saída impressa corresponda ao que a
rubrica pede.

### Entregável: captura do resultado de `model.evaluate(val_ds)`

In [ ]:
def evaluate(model, val_ds):
    """Equivalente PyTorch de model.evaluate(val_ds) do Keras: roda uma
    passada de avaliação completa e retorna [loss, accuracy]."""
    model.eval()
    correct, total, running_loss = 0, 0, 0.0
    with torch.no_grad():
        for images, labels in val_ds:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            running_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs, 1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)
    return [running_loss / total, correct / total]

resultado = evaluate(model, val_ds)
print("model.evaluate(val_ds) ->", resultado)
print(f"loss: {resultado[0]:.4f} - accuracy: {resultado[1]:.4f}")


## 12. Curva de acurácia por época

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(history['accuracy'], label='Treino')
plt.plot(history['val_accuracy'], label='Validação')
plt.title('Acurácia por época — MobileNetV2 Transfer Learning')
plt.xlabel('Época')
plt.ylabel('Acurácia')
plt.legend()
plt.grid(alpha=0.3)
plt.show()


## Resumo dos entregáveis desta atividade

1. **Link deste notebook Colab** executado com resultados visíveis (compartilhar como
   "Qualquer pessoa com o link").
2. **Print da célula 10** — acurácia final após 10 épocas.
3. **Print da célula 11** — resultado de `model.evaluate(val_ds)`.
4. **Print da célula 3 (estrutura)** — dataset organizado em `train/` e `validation/` com
   2 classes (`cats`, `dogs`).